# Entrainement - Prediction des types de panne

Objectif : construire un premier modele qui predit le type de panne futur a partir de la telemetrie TSDB.

Important : dans ce notebook, les labels de panne sont crees par des regles metier simples. Ce sont donc des labels synthetiques. Le modele apprendra surtout ces regles. C'est utile pour demarrer, mais pour un vrai modele il faudra idealement des pannes historiques reelles ou des scenarios simules clairement definis.

## 0. Installer les dependances si necessaire

In [ ]:
# A lancer une seule fois si besoin
# %pip install pandas matplotlib scikit-learn

## 1. Imports et configuration

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 100)

DUMP_PATH = Path("../tsdb_full_dump.sql")
assert DUMP_PATH.exists(), f"Dump introuvable: {DUMP_PATH.resolve()}"

DUMP_PATH.resolve()

## 2. Charger les tables utiles depuis le dump SQL

In [ ]:
def read_copy_table(dump_path: Path, table_name: str) -> pd.DataFrame:
    start = f"COPY public.{table_name} ("
    in_block = False
    columns = None
    rows = []

    with dump_path.open("r", encoding="utf-8") as file:
        for raw_line in file:
            line = raw_line.rstrip("\n")

            if not in_block and line.startswith(start):
                in_block = True
                columns_part = line.split("(", 1)[1].split(")", 1)[0]
                columns = [col.strip().strip('"') for col in columns_part.split(",")]
                continue

            if in_block:
                if line == r"\.":
                    break
                rows.append([None if value == r"\N" else value for value in line.split("\t")])

    if columns is None:
        raise ValueError(f"Table non trouvee dans le dump: {table_name}")

    return pd.DataFrame(rows, columns=columns)


sensor_data = read_copy_table(DUMP_PATH, "sensor_data")
sensor = read_copy_table(DUMP_PATH, "sensor")
server = read_copy_table(DUMP_PATH, "server")
fan = read_copy_table(DUMP_PATH, "fan")

sensor_data.shape, sensor.shape, server.shape, fan.shape

## 3. Convertir les types

In [ ]:
sensor_data["id"] = sensor_data["id"].astype(int)
sensor_data["time"] = pd.to_datetime(sensor_data["time"])
sensor_data["sensor_id"] = sensor_data["sensor_id"].astype(int)
sensor_data["value"] = sensor_data["value"].astype(float)

sensor["sensor_id"] = sensor["sensor_id"].astype(int)
sensor["server_id"] = sensor["server_id"].astype(int)
sensor["last_value"] = pd.to_numeric(sensor["last_value"], errors="coerce")

server["server_id"] = server["server_id"].astype(int)
server["cluster_id"] = server["cluster_id"].astype(int)
server["is_master"] = server["is_master"].map({"t": True, "f": False})
server["base_consumption_offset"] = pd.to_numeric(server["base_consumption_offset"], errors="coerce")

fan["fan_id"] = fan["fan_id"].astype(int)
fan["server_id"] = fan["server_id"].astype(int)
fan["speed_percent"] = fan["speed_percent"].astype(int)

## 4. Construire la table de telemetrie enrichie

On ajoute a chaque mesure son type de capteur, son unite et le serveur concerne.

In [ ]:
telemetry = (
    sensor_data
    .merge(sensor[["sensor_id", "server_id", "sensor_type", "unit"]], on="sensor_id", how="left")
    .merge(server[["server_id", "hostname", "is_master", "base_consumption_offset"]], on="server_id", how="left")
)

telemetry.head()

## 5. Passer en format large

Pour le machine learning, on veut une ligne par serveur et par timestamp.

In [ ]:
wide = telemetry.pivot_table(
    index=["time", "server_id", "hostname", "is_master", "base_consumption_offset"],
    columns="sensor_type",
    values="value",
    aggfunc="mean",
).reset_index()

wide.columns.name = None
wide = wide.sort_values(["server_id", "time"]).reset_index(drop=True)

wide.head()

## 6. Verifier les ventilateurs

Dans le dump actuel, `fan` represente les ventilateurs physiques. Les colonnes `FAN_SPEED_1` et `FAN_SPEED_2` sont des canaux de mesure dans la telemetrie.

In [ ]:
physical_fans = fan.groupby("server_id").agg(
    nb_physical_fans=("fan_id", "count"),
    current_fan_speed_mean=("speed_percent", "mean"),
).reset_index()

wide = wide.merge(physical_fans, on="server_id", how="left")
wide["avg_fan_speed"] = wide[["FAN_SPEED_1", "FAN_SPEED_2"]].mean(axis=1)

wide[["server_id", "hostname", "nb_physical_fans", "FAN_SPEED_1", "FAN_SPEED_2", "avg_fan_speed"]].head()

## 7. Creer les labels de panne adaptes a une temperature 30-50

Comme tu as change la temperature vers une plage `30-50`, des seuils fixes trop hauts peuvent produire presque uniquement du `normal`.

Pour un premier entrainement, on cree donc des labels synthetiques a partir des quantiles du dataset. Ce n'est pas une verite terrain reelle, mais ca permet d'avoir plusieurs classes pour entrainer et tester le pipeline.

Principe :

- temperature tres haute par rapport au dataset : `overheating`
- temperature haute + charge haute : `high_load_overheating`
- temperature haute + ventilateur bas : `fan_failure_possible`
- temperature haute + ventilateur deja haut : `cooling_underperformance`
- puissance tres haute avec charge pas tres haute : `power_anomaly`
- sinon : `normal`

In [ ]:
thresholds = {
    "temp_q60": wide["CPU_TEMP"].quantile(0.60),
    "temp_q70": wide["CPU_TEMP"].quantile(0.70),
    "temp_q80": wide["CPU_TEMP"].quantile(0.80),
    "temp_q90": wide["CPU_TEMP"].quantile(0.90),
    "fan_q35": wide["avg_fan_speed"].quantile(0.35),
    "fan_q65": wide["avg_fan_speed"].quantile(0.65),
    "load_q70": wide["LOAD"].quantile(0.70),
    "power_q90": wide["TOTAL_POWER"].quantile(0.90),
    "power_q95": wide["TOTAL_POWER"].quantile(0.95),
}

thresholds

In [ ]:
def label_current_failure(row) -> str:
    temp = row.get("CPU_TEMP")
    fan_speed = row.get("avg_fan_speed")
    load = row.get("LOAD")
    power = row.get("TOTAL_POWER")

    if pd.isna(temp) or pd.isna(fan_speed) or pd.isna(load):
        return "unknown"

    if pd.notna(power) and power >= thresholds["power_q95"] and load < thresholds["load_q70"]:
        return "power_anomaly"

    if temp >= thresholds["temp_q90"]:
        return "overheating"

    if temp >= thresholds["temp_q80"] and load >= thresholds["load_q70"]:
        return "high_load_overheating"

    if temp >= thresholds["temp_q70"] and fan_speed <= thresholds["fan_q35"]:
        return "fan_failure_possible"

    if temp >= thresholds["temp_q60"] and fan_speed >= thresholds["fan_q65"]:
        return "cooling_underperformance"

    return "normal"


wide["failure_type_now"] = wide.apply(label_current_failure, axis=1)
wide["failure_type_now"].value_counts().rename_axis("failure_type_now").reset_index(name="nb_points")

## 8. Creer la cible future

On ne veut pas predire la panne actuelle, mais la panne future.

Ici, on predit le type de panne au prochain timestamp du meme serveur. Si tes timestamps sont horaires, c'est une prediction a +1h. Si tes timestamps changent, adapte `PREDICTION_STEPS_AHEAD`.

In [ ]:
PREDICTION_STEPS_AHEAD = 1

wide["failure_type_future"] = (
    wide.groupby("server_id")["failure_type_now"]
    .shift(-PREDICTION_STEPS_AHEAD)
)

wide[["time", "server_id", "CPU_TEMP", "avg_fan_speed", "failure_type_now", "failure_type_future"]].head(10)

## 9. Creer des features temporelles

Un modele de panne doit voir les tendances : temperature qui monte, ventilateur qui ne repond pas, charge qui augmente, etc.

In [ ]:
for column in ["CPU_TEMP", "avg_fan_speed", "LOAD", "TOTAL_POWER"]:
    wide[f"{column}_delta"] = wide.groupby("server_id")[column].diff()
    wide[f"{column}_rolling_mean_3"] = (
        wide.groupby("server_id")[column]
        .rolling(3)
        .mean()
        .reset_index(level=0, drop=True)
    )
    wide[f"{column}_rolling_max_3"] = (
        wide.groupby("server_id")[column]
        .rolling(3)
        .max()
        .reset_index(level=0, drop=True)
    )

wide["hour"] = wide["time"].dt.hour
wide["dayofweek"] = wide["time"].dt.dayofweek
wide["is_master_int"] = wide["is_master"].astype(int)

wide.head()

## 10. Preparer le dataset modele

On garde uniquement les colonnes numeriques utiles comme entrees du modele.

In [ ]:
feature_columns = [
    "CPU_TEMP",
    "FAN_SPEED_1",
    "FAN_SPEED_2",
    "avg_fan_speed",
    "LOAD",
    "TOTAL_POWER",
    "nb_physical_fans",
    "base_consumption_offset",
    "CPU_TEMP_delta",
    "CPU_TEMP_rolling_mean_3",
    "CPU_TEMP_rolling_max_3",
    "avg_fan_speed_delta",
    "avg_fan_speed_rolling_mean_3",
    "avg_fan_speed_rolling_max_3",
    "LOAD_delta",
    "LOAD_rolling_mean_3",
    "LOAD_rolling_max_3",
    "TOTAL_POWER_delta",
    "TOTAL_POWER_rolling_mean_3",
    "TOTAL_POWER_rolling_max_3",
    "hour",
    "dayofweek",
    "is_master_int",
]

available_features = [col for col in feature_columns if col in wide.columns]

df_model = wide.dropna(subset=available_features + ["failure_type_future"]).copy()
df_model = df_model[df_model["failure_type_future"] != "unknown"]

df_model.shape, df_model["failure_type_future"].value_counts()

## 11. Split temporel train/test

Pour une serie temporelle, on evite de melanger aleatoirement le passe et le futur. On entraine sur le debut, puis on teste sur la fin.

In [ ]:
df_model = df_model.sort_values("time").reset_index(drop=True)

split_index = int(len(df_model) * 0.8)
train_df = df_model.iloc[:split_index]
test_df = df_model.iloc[split_index:]

X_train = train_df[available_features]
y_train = train_df["failure_type_future"]
X_test = test_df[available_features]
y_test = test_df["failure_type_future"]

train_df["time"].min(), train_df["time"].max(), test_df["time"].min(), test_df["time"].max()

## 12. Entrainer un premier modele baseline

On commence avec un Random Forest. Ce n'est pas forcement le meilleur modele final, mais c'est un bon point de depart pour verifier que le pipeline fonctionne.

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    min_samples_leaf=2,
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred, zero_division=0))

## 12 bis. Comparer plusieurs modeles

On compare plusieurs modeles avec les memes donnees train/test. La metrique principale ici est `macro_f1`, car elle donne le meme poids a chaque type de panne, meme si `normal` est beaucoup plus frequent.

In [ ]:
models = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "logistic_regression_balanced": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42),
    ),
    "random_forest_balanced": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        min_samples_leaf=2,
    ),
    "extra_trees_balanced": ExtraTreesClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        min_samples_leaf=2,
    ),
    "gradient_boosting": GradientBoostingClassifier(random_state=42),
}

model_results = []
fitted_models = {}
predictions = {}

for name, candidate in models.items():
    candidate.fit(X_train, y_train)
    pred = candidate.predict(X_test)

    fitted_models[name] = candidate
    predictions[name] = pred

    model_results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_test, pred, average="weighted", zero_division=0),
    })

results_df = pd.DataFrame(model_results).sort_values("macro_f1", ascending=False)
results_df

## 12 ter. Rapport du meilleur modele

On selectionne automatiquement le modele avec le meilleur `macro_f1`.

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_model_name]
best_pred = predictions[best_model_name]

print("Meilleur modele:", best_model_name)
print(classification_report(y_test, best_pred, zero_division=0))

## 13. Matrice de confusion du meilleur modele

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ConfusionMatrixDisplay.from_predictions(y_test, best_pred, xticks_rotation=45, ax=ax)
plt.tight_layout()
plt.show()

## 14. Importance des variables du Random Forest

Cette partie aide a comprendre ce que le Random Forest utilise le plus. Si le meilleur modele est une regression logistique ou un autre modele sans `feature_importances_`, on garde quand meme cette lecture sur le Random Forest.

In [ ]:
feature_importance = pd.DataFrame({
    "feature": available_features,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

feature_importance

In [ ]:
plt.figure(figsize=(10, 7))
plt.barh(feature_importance["feature"].head(15)[::-1], feature_importance["importance"].head(15)[::-1])
plt.title("Top 15 des variables les plus importantes")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 15. Inspecter les erreurs

On regarde les cas ou le modele se trompe. C'est souvent la partie la plus utile pour ameliorer les labels ou les features.

In [ ]:
errors = test_df.copy()
errors["prediction"] = best_pred
errors = errors[errors["failure_type_future"] != errors["prediction"]]

errors[[
    "time", "hostname", "CPU_TEMP", "avg_fan_speed", "LOAD", "TOTAL_POWER",
    "failure_type_now", "failure_type_future", "prediction"
]].head(30)

## 16. Sauvegarder le dataset d'entrainement

Cette sortie te permet de garder le dataset ML construit depuis le dump.

In [ ]:
OUTPUT_PATH = Path("training_dataset_failure_prediction.csv")
df_model.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH.resolve()

## Points a verifier apres execution

1. Si presque tout est `normal`, il faudra generer plus de scenarios de panne.
2. Si le score est tres haut, c'est normal avec des labels synthetiques : le modele apprend les seuils.
3. Si certaines classes n'apparaissent pas dans le test, il faut changer le split ou enrichir les donnees.
4. Pour une vraie prediction, la prochaine amelioration est de simuler explicitement des pannes : ventilateur bloque, refroidissement faible, surchauffe longue, puissance anormale, capteur bloque.